In [2]:
import numpy as np
np.set_printoptions(suppress=True)

from sklearn.datasets import load_diabetes

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression

In [3]:
x, y = load_diabetes(return_X_y = True)

print(x[0:5])
print()
print()

print(y[0:5])

[[ 0.03807591  0.05068012  0.06169621  0.02187239 -0.0442235  -0.03482076
  -0.04340085 -0.00259226  0.01990749 -0.01764613]
 [-0.00188202 -0.04464164 -0.05147406 -0.02632753 -0.00844872 -0.01916334
   0.07441156 -0.03949338 -0.06833155 -0.09220405]
 [ 0.08529891  0.05068012  0.04445121 -0.00567042 -0.04559945 -0.03419447
  -0.03235593 -0.00259226  0.00286131 -0.02593034]
 [-0.08906294 -0.04464164 -0.01159501 -0.03665608  0.01219057  0.02499059
  -0.03603757  0.03430886  0.02268774 -0.00936191]
 [ 0.00538306 -0.04464164 -0.03638469  0.02187239  0.00393485  0.01559614
   0.00814208 -0.00259226 -0.03198764 -0.04664087]]


[151.  75. 141. 206. 135.]


In [4]:
# TrainTest Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

# Creating our `StochasticGD` class

In [5]:
class Custom_SGDRegressor:
    
    def __init__(self,learning_rate=0.01,epochs=100):
        
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        # We are not printing derivatives here bcz here in SGD, we will have derivatives in thousands.
        
    def fit(self,x_train,y_train):
        # init your coefs
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])
        
        for i in range(self.epochs):
            for j in range(x_train.shape[0]):
                idx = np.random.randint(0,x_train.shape[0])
                
                y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_
                
                intercept_der = -2 * (y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)
                
                coef_der = -2 * np.dot((y_train[idx] - y_hat),x_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)
        
            print(f"""{'='*15} Epoch {i} {'='*15}
        Coeffecients: {self.coef_}
        
        Intercept: {self.intercept_}
        
        """)
            
        print(f"""{'='*15} Final Result {'='*15}
        Coeffecients: {self.coef_}
        Intercept: {self.intercept_}
        """)
    
    def predict(self,x_test):
        return np.dot(x_test,self.coef_) + self.intercept_

In [6]:
sgd = Custom_SGDRegressor(epochs=1500,learning_rate=0.1)
sgd.fit(x_train,y_train)

=============== Epoch 0 ===============
        Coeffecients: [ 33.14702031 -16.36368332 122.94815197  71.56172963  28.97582378
  15.24727773 -75.98694765  78.56827152 117.47063145  63.38648222]

        Intercept: 143.50173494310246

        
=============== Epoch 1 ===============
        Coeffecients: [  47.69577362  -23.70438847  211.81380507  136.36573375   22.44388854
    0.74317983 -122.40010925  107.20094903  182.12645975   98.68936802]

        Intercept: 151.7214166879701

        
=============== Epoch 2 ===============
        Coeffecients: [  64.92808632  -48.55913367  277.41566522  170.59995715   28.16015193
   -0.10466916 -147.97289398  135.52983695  228.25492807  130.22407137]

        Intercept: 132.52430946915402

        
=============== Epoch 3 ===============
        Coeffecients: [  69.38565675  -55.502725    334.08015974  213.67734963   19.29977151
  -16.46184343 -167.07029054  145.74136802  267.0542141   146.00180935]

        Intercept: 181.74723240818477

    

# Comparision:  LinearRegression and SGDRegressor results

In [7]:
lr = LinearRegression()

lr.fit(x_train, y_train)

y_pred_lr = lr.predict(x_test)
y_pred_sgd = sgd.predict(x_test)

r2_lr = r2_score(y_test, y_pred_lr)
r2_sgd = r2_score(y_test, y_pred_sgd)

print(f"""{'='*15} LinearRegression {'='*15}
-> Co-effecients: {lr.coef_}
-> Intercept: {lr.intercept_}
-> R2_Score: {r2_lr}

{'='*15} StochasticGD {'='*15}
-> Co-effecients: {sgd.coef_}
-> Intercept: {sgd.intercept_}
-> R2_Score: {r2_sgd}""")



=============== LinearRegression ===============
-> Co-effecients: [ -30.62219046 -272.2506057   528.85252681  327.69716891 -581.01973841
  332.97061365  -27.97314298  139.28036115  665.07667773   61.90517166]
-> Intercept: 151.4300932966009
-> R2_Score: 0.4384316213369279

=============== StochasticGD ===============
-> Co-effecients: [ -37.42462325 -293.61020565  529.43258239  293.94654615 -524.80602126
  296.77214474  -46.37827173  138.29608355  635.62264449  102.58440879]
-> Intercept: 135.6907090100792
-> R2_Score: 0.36506935703849186


# SGDRegressor by Sklearn

In [8]:
from sklearn.linear_model import SGDRegressor

In [9]:
sgd_2 = SGDRegressor(
    loss = 'squared_error',
    max_iter = 1500,
    tol = 0.00001,
    n_iter_no_change = 5,
    early_stopping  = True,
    validation_fraction = 0.1,
    verbose = 1,
    learning_rate = 'invscaling',
    random_state = 1
)

sgd_2.fit(x_train, y_train)
y_pred_2 = sgd_2.predict(x_test)

# Check difference in prediction from our class and scikit learn's class.
# diff = np.mean(y_pred_sgd - y_pred_2)**2
diff = np.mean((y_pred_sgd - y_pred_2)**2)

r2_sgd_2 = r2_score(y_test, y_pred_2)

-- Epoch 1
Norm: 4.49, NNZs: 10, Bias: 96.779595, T: 317, Avg. loss: 7433.554818
Total training time: 0.00 seconds.
-- Epoch 2
Norm: 7.69, NNZs: 10, Bias: 124.552391, T: 634, Avg. loss: 3862.658697
Total training time: 0.00 seconds.
-- Epoch 3
Norm: 10.43, NNZs: 10, Bias: 137.857671, T: 951, Avg. loss: 3224.138679
Total training time: 0.00 seconds.
-- Epoch 4
Norm: 12.94, NNZs: 10, Bias: 144.543894, T: 1268, Avg. loss: 3049.753425
Total training time: 0.01 seconds.
-- Epoch 5
Norm: 15.28, NNZs: 10, Bias: 148.087865, T: 1585, Avg. loss: 2991.363388
Total training time: 0.01 seconds.
-- Epoch 6
Norm: 17.49, NNZs: 10, Bias: 149.930118, T: 1902, Avg. loss: 2966.578574
Total training time: 0.01 seconds.
-- Epoch 7
Norm: 19.59, NNZs: 10, Bias: 151.151308, T: 2219, Avg. loss: 2951.523594
Total training time: 0.01 seconds.
-- Epoch 8
Norm: 21.61, NNZs: 10, Bias: 152.126904, T: 2536, Avg. loss: 2939.666977
Total training time: 0.01 seconds.
-- Epoch 9
Norm: 23.56, NNZs: 10, Bias: 152.449760, T:

c:\Users\jugal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1579: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [10]:
print(f"""{'='*15} Scikit learn's SGDRegressor {'='*15}
-> Co-effecients: {sgd_2.coef_}
-> Intercept: {sgd_2.intercept_}
-> R2_Score: {r2_sgd_2}

{'='*15} Custom_SGD {'='*15}
-> Co-effecients: {sgd.coef_}
-> Intercept: {sgd.intercept_}
-> R2_Score: {r2_sgd}

{'='*15} Mean Error in Prediction from both the classes {'='*15}
{diff}
""")

=============== Scikit learn's SGDRegressor ===============
-> Co-effecients: [  52.77858291  -50.07735896  300.26015935  194.81180512   31.75328271
   -1.67376306 -160.97393829  141.59374041  248.7563431   145.34342114]
-> Intercept: [151.72025467]
-> R2_Score: 0.3913301526050277

=============== Custom_SGD ===============
-> Co-effecients: [ -37.42462325 -293.61020565  529.43258239  293.94654615 -524.80602126
  296.77214474  -46.37827173  138.29608355  635.62264449  102.58440879]
-> Intercept: 135.6907090100792
-> R2_Score: 0.36506935703849186

=============== Mean Error in Prediction from both the classes ===============
718.3421672935281



* If we increase  `tol` it is converging very early and not coming even near to the Custom Class's results.
* It is only converging proper at 1500 epochs and `tol = 0.00001`

* Here, in total, we will have **353-10%(Validation) = 317*1500 = 475500 updates**

In [11]:
y_pred_sgd[0:5]

array([104.02177807,  95.85378469, 171.51774162,  54.20040641,
       155.10664797])

In [12]:
y_pred_2[0:5]

array([129.11045347,  91.81688888, 167.37391395,  80.20576188,
       180.78626682])

In [13]:
y_test[0:5]

array([ 78., 152., 200.,  59., 311.])